In [13]:
import pandas as pd
import numpy as np
from krippendorff_alpha.compute_alpha import compute_alpha
from krippendorff_alpha.preprocessing import ColumnMapping

In [14]:
# ---------------------------------------------------------
# 1. Load the Excel Workbook
# ---------------------------------------------------------
excel_path = "/home/david/Git/SDLT_Number_Crunching/Data/adiya_isla_scores.xlsx"

# List of sheets to evaluate (skipping overview sheets like 'Whole' if present)
sheets_to_process = [
    "CoC", "Au", "In", "Re", "La", "Fi", "Cq", "Rr", "Pc"
]

# ---------------------------------------------------------
# 2. Configuration Settings
# ---------------------------------------------------------
annotator_cols = ["annotator1", "annotator2", "annotator3"]
scale = ["0", "1", "2"]

column_mapping = ColumnMapping(
    text_col="Publication",
    annotator_cols=annotator_cols
)

In [17]:
def compute_raw_agreement(df_annotators: pd.DataFrame) -> float:
    """Computes percent raw agreement (unanimous agreement across all raters)."""
    # Checks if all annotators in a row gave the identical score
    unanimous = df_annotators.nunique(axis=1) == 1
    return float(unanimous.mean() * 100.0)


def evaluate_inter_rater_reliability(
    excel_path: str,
    sheets_to_process: list[str],
    annotator_cols: list[str],
    column_mapping: dict,
    scale: list[str] = ["0", "1", "2"],
    compute_alpha_fn=None,
    n_bootstraps: int = 1000,
    ci_level: float = 0.95,
    random_state: int = 42,
) -> pd.DataFrame:
    """Computes Krippendorff's Alpha, Bootstrapped 95% CIs, and Raw Agreement

    for multi-round annotation sheets.
    """
    if compute_alpha_fn is None:
        raise ValueError("Please provide a valid `compute_alpha` function.")

    rng = np.random.default_rng(random_state)
    xls = pd.ExcelFile(excel_path)
    results_list = []

    alpha_tail = (1 - ci_level) / 2.0
    lower_pct = alpha_tail * 100
    upper_pct = (1 - alpha_tail) * 100

    for sheet_name in sheets_to_process:
        if sheet_name not in xls.sheet_names:
            print(
                f"Warning: Sheet '{sheet_name}' not found in workbook. Skipping."
            )
            continue

        df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=[0, 1])
        pub_col_name = [
            col for col in df_raw.columns if "Publication" in str(col)
        ][0]
        publication_series = df_raw[pub_col_name]

        row_result = {"Category": sheet_name}

        for round_name in ["Round 1", "Round 2"]:
            round_suffix = round_name.replace(" ", "_")

            df_round = df_raw[round_name].copy()
            df_round["Publication"] = publication_series.values
            df_round[annotator_cols] = df_round[annotator_cols].astype(str)

            # # 1. Percent Raw Agreement
            # raw_agree = compute_raw_agreement(df_round[annotator_cols])
            # row_result[f"Raw_Agreement_{round_suffix}"] = raw_agree

            # 2. Point Estimate Alpha
            res = compute_alpha_fn(
                df=df_round,
                data_type="ordinal",
                column_mapping=column_mapping,
                ordinal_scale=scale,
            )
            point_alpha = res["alpha"]
            row_result[f"Alpha_{round_suffix}"] = point_alpha

            # 3. Non-Parametric Bootstrapping (resampling publication units)
            n_units = len(df_round)
            boot_alphas = []

            for i in range(n_bootstraps):
                # Sample rows (units/publications) with replacement

                if i % 100 == 0:
                    print(
                        f"Bootstrapping {sheet_name} - {round_name}: Iteration {i}/{n_bootstraps}"
                    )

                boot_indices = rng.choice(n_units, size=n_units, replace=True)
                df_boot = df_round.iloc[boot_indices].reset_index(drop=True)

                try:
                    boot_res = compute_alpha_fn(
                        df=df_boot,
                        data_type="ordinal",
                        column_mapping=column_mapping,
                        ordinal_scale=scale,
                    )
                    boot_alphas.append(boot_res["alpha"])
                except Exception:
                    continue  # Handles edge cases where resample has zero variance

            if len(boot_alphas) > 0:
                ci_low = np.percentile(boot_alphas, lower_pct)
                ci_high = np.percentile(boot_alphas, upper_pct)
            else:
                ci_low, ci_high = np.nan, np.nan

            row_result[f"Alpha_{round_suffix}_CI_Low"] = ci_low
            row_result[f"Alpha_{round_suffix}_CI_High"] = ci_high

        results_list.append(row_result)

    return pd.DataFrame(results_list)

In [ ]:
# ---------------------------------------------------------
# 4. Format & Display Results
# ---------------------------------------------------------
df_results = evaluate_inter_rater_reliability(
    excel_path=excel_path,
    sheets_to_process=sheets_to_process,
    annotator_cols=annotator_cols,
    column_mapping=column_mapping,
    scale=["0", "1", "2"],
    compute_alpha_fn=compute_alpha,
    n_bootstraps=1000,
    ci_level=0.95,
    random_state=42
)

print("\n--- Krippendorff's Alpha Results ---")
print(df_results.to_string(index=False))

Bootstrapping CoC - Round 1: Iteration 0/1000
Bootstrapping CoC - Round 1: Iteration 100/1000
Bootstrapping CoC - Round 1: Iteration 200/1000
Bootstrapping CoC - Round 1: Iteration 300/1000
Bootstrapping CoC - Round 1: Iteration 400/1000
Bootstrapping CoC - Round 1: Iteration 500/1000
Bootstrapping CoC - Round 1: Iteration 600/1000
Bootstrapping CoC - Round 1: Iteration 700/1000
Bootstrapping CoC - Round 1: Iteration 800/1000
Bootstrapping CoC - Round 1: Iteration 900/1000
Bootstrapping CoC - Round 2: Iteration 0/1000
Bootstrapping CoC - Round 2: Iteration 100/1000
Bootstrapping CoC - Round 2: Iteration 200/1000
Bootstrapping CoC - Round 2: Iteration 300/1000
Bootstrapping CoC - Round 2: Iteration 400/1000
Bootstrapping CoC - Round 2: Iteration 500/1000
Bootstrapping CoC - Round 2: Iteration 600/1000
Bootstrapping CoC - Round 2: Iteration 700/1000
Bootstrapping CoC - Round 2: Iteration 800/1000
Bootstrapping CoC - Round 2: Iteration 900/1000
Bootstrapping Au - Round 1: Iteration 0/1000

KeyboardInterrupt: 